# BirdCLEF+ 2026 — Inference & Submission

V7b 简化基线模型推理。流程: 声景文件 → 12×5s 窗口 → torchaudio mel → 模型预测 → 后处理 → submission.csv

In [ ]:
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

USE_AMP = torch.cuda.is_available()
try:
    from torch.amp import autocast
    AMP_DEVICE = 'cuda'
except ImportError:
    from torch.cuda.amp import autocast
    AMP_DEVICE = None

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, Device: {DEVICE}')

In [ ]:
# ── 路径配置 ──
IS_KAGGLE = Path('/kaggle/input').exists()

def find_data_dir():
    if not IS_KAGGLE: return Path('../data/raw')
    for p in Path('/kaggle/input').iterdir():
        if (p / 'test_soundscapes').exists(): return p
        if p.is_dir():
            for sub in p.iterdir():
                if sub.is_dir() and (sub / 'test_soundscapes').exists(): return sub
    raise FileNotFoundError('test_soundscapes not found')

def find_model_dir():
    """Search for model weights in multiple locations."""
    candidates = [
        Path('/kaggle/input/birdclef2026-baseline-b0-training'),
        Path('/kaggle/input/birdclef2026-weights'),
    ]
    if IS_KAGGLE:
        for p in Path('/kaggle/input').iterdir():
            if list(p.glob('best_fold*.pth')):
                return p
            for sub in p.iterdir():
                if sub.is_dir() and list(sub.glob('best_fold*.pth')):
                    return sub
    for c in candidates:
        if c.exists() and list(c.glob('best_fold*.pth')):
            return c
    return Path('../models')

DATA_DIR = find_data_dir()
MODEL_DIR = find_model_dir()
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('../models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data: {DATA_DIR}')
print(f'Models: {MODEL_DIR}')
print(f'Model files: {list(MODEL_DIR.glob("best_fold*.pth"))}')

In [ ]:
# ── 常量 ──
SR = 32_000
CLIP_DURATION = 5.0
CLIP_SAMPLES = int(SR * CLIP_DURATION)
N_FFT = 1024
HOP_LENGTH = 320
N_MELS = 128
FMIN = 50
FMAX = 14000
NUM_CLASSES = 234

submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', nrows=0)
SPECIES_COLS = [c for c in submission.columns if c != 'row_id']
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
TAX_MAP = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name']))
LABEL2IDX = {label: i for i, label in enumerate(SPECIES_COLS)}
print(f'Species: {len(SPECIES_COLS)}')

In [ ]:
# ── 音频处理（torchaudio，与训练一致） ──
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
AMP_TO_DB = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)
RESAMPLERS = {}

def load_audio(path, sr=SR, duration=CLIP_DURATION, offset=0.0):
    tl = int(sr * duration)
    try:
        waveform, file_sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if file_sr != sr:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, sr)
            waveform = RESAMPLERS[file_sr](waveform)
        start = int(offset * sr)
        waveform = waveform[:, start:start + tl].squeeze(0)
    except Exception:
        waveform = torch.zeros(tl)
    if waveform.numel() == 0:
        waveform = torch.zeros(tl)
    if waveform.numel() < tl:
        reps = (tl // waveform.numel()) + 1
        waveform = waveform.repeat(reps)[:tl]
    return waveform[:tl]

def audio_to_melspec(waveform):
    mel = AMP_TO_DB(MEL_TRANSFORM(waveform.unsqueeze(0))).squeeze(0)
    return (mel - mel.mean()) / (mel.std() + 1e-6)

# 验证
test_wav = torch.randn(CLIP_SAMPLES)
test_mel = audio_to_melspec(test_wav)
print(f'Mel test: {test_mel.shape}')

In [ ]:
# ── 模型（与 V7b 训练一致） ──
class BirdCLEFB0(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b0_ns', pretrained=False,
            in_chans=1, num_classes=0, global_pool='avg')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(1280, NUM_CLASSES)kaggle kernels output montyeternity/birdclef2026-baseline-b0-training -p /path/to/dest

    def forward(self, mel):
        return self.fc(self.dropout(self.backbone(mel)))

# 加载权重
models = []
for p in sorted(MODEL_DIR.glob('best_fold*.pth')):
    m = BirdCLEFB0().to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE, weights_only=True))
    m.eval()
    models.append(m)
    print(f'  Loaded: {p.name}')
print(f'Total models: {len(models)}')
if not models:
    raise FileNotFoundError(f'No model weights found in {MODEL_DIR}')

In [ ]:
# ── 后处理 ──
TIME_PRIOR = {
    'Aves':     [0.2,0.2,0.3,0.5,0.9,1.0,1.0,0.8,0.6,0.4,0.3,0.3,
                 0.3,0.3,0.3,0.4,0.5,0.8,1.0,0.6,0.3,0.2,0.2,0.2],
    'Amphibia': [0.8,0.8,0.7,0.5,0.3,0.2,0.2,0.1,0.1,0.1,0.1,0.1,
                 0.1,0.1,0.1,0.1,0.2,0.4,0.8,1.0,1.0,1.0,0.9,0.9],
    'Insecta':  [0.5,0.5,0.5,0.8,0.6,0.4,1.0,1.0,0.4,0.3,0.3,0.3,
                 0.3,0.3,0.3,0.3,0.3,0.4,0.5,0.8,0.5,0.5,0.5,0.5],
}

SONOTYPE_GROUPS = {
    'A': ['47158son08','47158son11','47158son20'],
    'B': ['47158son13','47158son22','47158son23'],
    'C': ['47158son15','47158son16','47158son25'],
    'D': ['47158son04','47158son10'],
}

# 共现矩阵
cond_prob = np.zeros((NUM_CLASSES, NUM_CLASSES))
labels_file = DATA_DIR / 'train_soundscapes_labels.csv'
if labels_file.exists():
    labels_df = pd.read_csv(labels_file)
    for _, row in labels_df.iterrows():
        idxs = [LABEL2IDX[l.strip()] for l in str(row.get('primary_label','')).split(';')
                if l.strip() in LABEL2IDX]
        for a in idxs:
            for b in idxs:
                if a != b: cond_prob[a][b] += 1
    cond_prob /= (cond_prob.sum(axis=1, keepdims=True) + 1e-8)
    print(f'Co-occurrence: {(cond_prob > 0).sum()} non-zero entries')
else:
    print('No train_soundscapes_labels.csv, skipping co-occurrence')

def parse_hour(filename):
    parts = Path(filename).stem.split('_')
    if len(parts) >= 6:
        try: return int(parts[5][:2])
        except: pass
    return 12

def postprocess(preds, hour):
    p = preds.copy()
    detected = np.where(p > 0.5)[0]
    for a in detected:
        for b in range(len(p)):
            if cond_prob[a][b] > 0.3:
                p[b] = max(p[b], p[a] * cond_prob[a][b] * 0.15)
    h = hour % 24
    for idx, col in enumerate(SPECIES_COLS):
        cls = TAX_MAP.get(col, '')
        if cls in TIME_PRIOR:
            f = TIME_PRIOR[cls][h]
            p[idx] = p[idx] * 0.9 + p[idx] * f * 0.1
    col2idx = {c: i for i, c in enumerate(SPECIES_COLS)}
    for members in SONOTYPE_GROUPS.values():
        mi = [col2idx[m] for m in members if m in col2idx]
        if mi:
            mx = max(p[i] for i in mi)
            for i in mi:
                p[i] = mx / len(mi)
    return p

In [ ]:
# ── 推理 ──
def _amp_ctx():
    return autocast(AMP_DEVICE) if AMP_DEVICE else autocast()

test_dir = DATA_DIR / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

rows = []
for filepath in tqdm(test_files, desc='Inference'):
    stem = filepath.stem
    hour = parse_hour(str(filepath))

    # 预加载整个 60s 音频一次
    try:
        full_wav, file_sr = torchaudio.load(str(filepath))
        if full_wav.shape[0] > 1:
            full_wav = full_wav.mean(dim=0, keepdim=True)
        if file_sr != SR:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, SR)
            full_wav = RESAMPLERS[file_sr](full_wav)
        full_wav = full_wav.squeeze(0)
    except Exception:
        full_wav = torch.zeros(SR * 60)

    for start_sec in range(0, 60, 5):
        start_sample = start_sec * SR
        clip = full_wav[start_sample:start_sample + CLIP_SAMPLES]
        if clip.numel() < CLIP_SAMPLES:
            pad = torch.zeros(CLIP_SAMPLES - clip.numel())
            clip = torch.cat([clip, pad])

        mel = audio_to_melspec(clip)
        mel_t = mel.unsqueeze(0).unsqueeze(0).to(DEVICE)

        fold_preds = []
        for model in models:
            with torch.no_grad():
                if USE_AMP:
                    with _amp_ctx(): logits = model(mel_t)
                else:
                    logits = model(mel_t)
            fold_preds.append(torch.sigmoid(logits).cpu().numpy()[0])
        avg_pred = np.mean(fold_preds, axis=0)
        avg_pred = postprocess(avg_pred, hour)

        row_id = f'{stem}_{start_sec + 5}'
        row = {'row_id': row_id}
        for i, col in enumerate(SPECIES_COLS):
            row[col] = float(avg_pred[i])
        rows.append(row)

sub = pd.DataFrame(rows)
sub.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'\nSubmission: {len(sub)} rows, {len(sub.columns)} columns')
print(f'Saved to: {OUTPUT_DIR / "submission.csv"}')
sub.head()

In [ ]:
# ── 提交验证 ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert list(sub.columns) == list(sample_sub.columns), 'Column mismatch!'
assert len(sub) == len(sample_sub), f'Row count mismatch: {len(sub)} vs {len(sample_sub)}'
assert sub['row_id'].tolist() == sample_sub['row_id'].tolist(), 'row_id mismatch!'

pred_cols = [c for c in sub.columns if c != 'row_id']
pred_vals = sub[pred_cols].values
print(f'Pred range: [{pred_vals.min():.6f}, {pred_vals.max():.6f}]')
print(f'Pred mean: {pred_vals.mean():.6f}')
print(f'Non-zero (>0.01): {(pred_vals > 0.01).sum()} / {pred_vals.size} ({(pred_vals > 0.01).mean()*100:.2f}%)')
print('\nSubmission format OK!')